# SEM-NAS — NAS-Bench-201 experiments (online proxy compute)

Reference notebook for the paper *Budgeted Fixed-Proxy Search for Zero-Shot NAS on NAS-Bench-201 via Sample-Efficient Memetic NAS*.

Running this notebook from top to bottom is sufficient to reproduce a SEM-NAS experiment end-to-end:

1. **Phase 0** — install Python dependencies and check the device.
2. **Phase 1** — auto-download every external resource the experiments need (NAS-Bench-201 release, CIFAR-10, CIFAR-100). After this phase the rest of the notebook is offline.
3. **Phase 2** — run SEM-NAS at FFC = 100 with **online** zero-cost proxy compute. Each candidate triggers a real PyTorch proxy run on the actual NB-201 architecture — there are no precomputed proxy pickles.
4. **Phase 3** — compare SEM-NAS to a budgeted fixed-proxy baseline (Generic GA) on the same cell.
5. **Phase 4** — mini-matrix sketch following the paper's 84-cell layout.

Optional: ImageNet-16-120 is not bundled in torchvision and must be downloaded separately from the NAS-Bench-201 README; CIFAR-10 / CIFAR-100 cells run without it.

---
## Phase 0: install dependencies

In [ ]:
%pip install -q numpy torch torchvision gdown nas-bench-201 matplotlib pandas

In [ ]:
import sys, time
from pathlib import Path

# Make code/ importable regardless of where the notebook is launched.
REPO = Path.cwd().resolve()
while REPO.name != 'code' and REPO.parent != REPO:
    REPO = REPO.parent
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

import numpy as np
import torch

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('repo root :', REPO)
print('torch     :', torch.__version__, '| device:', DEVICE)

---
## Phase 1: download all required data

This phase is idempotent — already-downloaded files are reused. Three resources are fetched:

| Resource | Size | Source |
|---|---|---|
| `NAS-Bench-201-v1_1-096897.pth` | ~2.2 GB | Google Drive (gdown) |
| CIFAR-10 train images | ~170 MB | torchvision |
| CIFAR-100 train images | ~170 MB | torchvision |

On first run this can take several minutes; subsequent runs reuse the local copies.

Output locations:
* NB-201 release → `code/data/nb201/NAS-Bench-201-v1_1-096897.pth`
* CIFAR-10/100   → `code/data/torchvision/cifar-10-batches-py/` and `cifar-100-python/`

In [ ]:
# 1a) NAS-Bench-201 .pth (auto-download, idempotent)
from scripts.download_nb201 import ensure_nb201_api
from sem_nas.proxy import NB201Api

api_path = ensure_nb201_api()
print('NB-201 API :', api_path)

# 1b) CIFAR-10 + CIFAR-100 (torchvision, idempotent)
import torchvision
TV_ROOT = REPO / 'data' / 'torchvision'
TV_ROOT.mkdir(parents=True, exist_ok=True)

print('CIFAR-10  : checking / downloading...')
_ = torchvision.datasets.CIFAR10(root=str(TV_ROOT), train=True, download=True)
print('  ->', TV_ROOT / 'cifar-10-batches-py')

print('CIFAR-100 : checking / downloading...')
_ = torchvision.datasets.CIFAR100(root=str(TV_ROOT), train=True, download=True)
print('  ->', TV_ROOT / 'cifar-100-python')

# 1c) Preload NB-201 API for the datasets we'll actually search over.
DATASETS_TO_PRELOAD = ('cifar10', 'cifar100')
print(f'\nLoading NB-201 trained-accuracy table for {DATASETS_TO_PRELOAD} '
      '(scans 15,625 archs once)...')
nb201_api = NB201Api(api_path, datasets=DATASETS_TO_PRELOAD)
for ds in DATASETS_TO_PRELOAD:
    accs = nb201_api.test_accuracy_array(ds)
    print(f'  {ds:<8} oracle best = {np.nanmax(accs):.2f}%  '
          f'(mean = {np.nanmean(accs):.2f}%)')

print('\nAll required data is ready. The remaining phases run fully offline.')

**Optional: ImageNet-16-120**

ImageNet-16-120 (~3.7 GB) is not bundled in torchvision. Download the NAS-Bench-201 ImageNet16 archive manually from
[the NAS-Bench-201 README](https://github.com/D-X-Y/NAS-Bench-201) and unpack it under `code/data/ImageNet16/`. The CIFAR-10 / CIFAR-100 experiments below run without it; uncomment the cell to enable IN16-120 batches.

In [ ]:
# Optional: load ImageNet-16-120 (only if you have unpacked the NB-201 archive).
#
# IN16_ROOT = REPO / 'data' / 'ImageNet16'
# if IN16_ROOT.exists():
#     nb201_api = NB201Api(api_path,
#                          datasets=('cifar10', 'cifar100', 'imagenet16_120'))
#     print('ImageNet-16-120 root found:', IN16_ROOT)
# else:
#     print('ImageNet-16-120 root not found — skipping. Use cifar10 / cifar100 below.')

---
## Phase 2: run SEM-NAS at FFC = 100, online proxy compute

Every candidate produced by SEM-NAS
1. instantiates the actual NAS-Bench-201 cell network from the length-6 encoding,
2. initializes its weights with a deterministic seed,
3. runs the chosen zero-cost proxy on a fixed cached minibatch (sampled once at backend construction),
4. returns a single scalar score.

We use the paper-final hyperparameters (`pop_size=10, K_gen=4, K_LS=3, W=3, b_LS=25, mutation_prob=1/L`).

In [ ]:
from sem_nas.evaluator import FitnessEvaluator
from sem_nas.proxy import OnlineProxyBackend
from sem_nas.proxy.proxies import PROXY_NAMES
from sem_nas.sem_nas import run as run_sem_nas
from sem_nas.baselines import BASELINES

DATASET = 'cifar10'      # one of cifar10 / cifar100 (or imagenet16_120 if enabled above)
PROXY   = 'synflow'      # data-free, fastest of the seven
FFC     = 100
SEED    = 0

# cells_per_stage=5 reproduces the full NB-201 backbone. Drop to 2 or 3
# on CPU laptops to keep the per-FFC wall time low.
CELLS_PER_STAGE = 3 if DEVICE == 'cpu' else 5

backend = OnlineProxyBackend(
    proxy_name=PROXY,
    dataset=DATASET,
    batch_size=16,
    n_batches=2 if PROXY == 'zico' else 1,
    device=DEVICE,
    data_source='torchvision',
    torchvision_root=str(TV_ROOT),
    init_seed=SEED,
    cell_seed=SEED,
    cells_per_stage=CELLS_PER_STAGE,
    nb201_api=nb201_api,
)
evaluator = FitnessEvaluator(backend, max_evals=FFC)
np.random.seed(SEED)

t0 = time.time()
best_arch, pop, fit = run_sem_nas(evaluator)
elapsed = time.time() - t0

print(f'SEM-NAS / {DATASET} / {PROXY}, FFC = {evaluator.ffc}')
print(f'  best encoding   : {best_arch.tolist()}')
print(f'  best proxy score: {max(evaluator.best_score_per_ffc):.4f}')
print(f'  test accuracy   : {evaluator.get_test_accuracy(best_arch):.2f}%')
print(f'  wall time       : {elapsed:.1f}s '
      f'(~{elapsed / max(evaluator.ffc, 1) * 1000:.0f} ms / FFC, online compute)')

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6.5, 3.2))
ax.plot(np.arange(1, len(evaluator.best_score_per_ffc) + 1),
        evaluator.best_score_per_ffc, color='#d62728', linewidth=2.0,
        marker='*', markersize=4)
ax.set_xlabel('FFC')
ax.set_ylabel(f'best {PROXY} score (running)')
ax.set_title(f'SEM-NAS / {DATASET} / {PROXY} '
             f'(online proxy compute, seed={SEED})')
ax.grid(True, linewidth=0.3, alpha=0.5)
plt.tight_layout()
plt.show()

---
## Phase 3: SEM-NAS vs Generic GA at the same FFC budget

Same `(dataset, proxy, seed)` cell; only the search loop differs.

In [ ]:
def run_method(method_name, *, proxy=PROXY, dataset=DATASET, ffc=FFC, seed=SEED):
    bk = OnlineProxyBackend(
        proxy_name=proxy, dataset=dataset,
        batch_size=16, n_batches=2 if proxy == 'zico' else 1,
        device=DEVICE, data_source='torchvision',
        torchvision_root=str(TV_ROOT),
        init_seed=seed, cell_seed=seed,
        cells_per_stage=CELLS_PER_STAGE,
        nb201_api=nb201_api,
    )
    ev = FitnessEvaluator(bk, max_evals=ffc)
    np.random.seed(seed)
    if method_name == 'sem_nas':
        best, _, _ = run_sem_nas(ev)
    else:
        best = BASELINES[method_name](ev)
    return ev, best

results = {}
for method in ('sem_nas', 'generic_ga'):
    t0 = time.time()
    ev, best = run_method(method)
    results[method] = (ev, best, time.time() - t0)

print(f'{"method":<14} {"best proxy":>14} {"test acc":>10} {"time (s)":>10}')
for m, (ev, best, dt) in results.items():
    print(f'{m:<14} {max(ev.best_score_per_ffc):>14.4f} '
          f'{ev.get_test_accuracy(best):>9.2f}% {dt:>10.1f}')

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 3.2))
styles = {
    'sem_nas':   {'color': '#d62728', 'linestyle': '-',  'marker': '*',
                   'label': 'Proposed (SEM-NAS)'},
    'generic_ga': {'color': '#ff7f0e', 'linestyle': (0, (3, 1, 1, 1)),
                   'marker': 'D', 'label': 'Generic GA'},
}
for method, (ev, _, _) in results.items():
    ax.plot(np.arange(1, len(ev.best_score_per_ffc) + 1),
            ev.best_score_per_ffc, linewidth=1.8, markersize=4, **styles[method])
ax.set_xlabel('FFC')
ax.set_ylabel(f'best {PROXY} score (running)')
ax.set_title(f'SEM-NAS vs Generic GA / {DATASET} / {PROXY} '
             '(online compute, single seed)')
ax.grid(True, linewidth=0.3, alpha=0.5)
ax.legend(loc='lower right', frameon=True)
plt.tight_layout()
plt.show()

---
## Phase 4: small mini-matrix

A handful of cells in the same layout as the paper's 84-cell main matrix. Bump `SEEDS`, `METHODS`, and `PROXIES` to scale up. Every cell triggers real PyTorch proxy compute per FFC, so wall time scales accordingly; consider `scripts/run_main_matrix.py --workers 8` for parallel runs.

In [ ]:
import itertools
import pandas as pd

METHODS = ('sem_nas', 'generic_ga')
PROXIES = ('synflow', 'snip')
DATASETS = ('cifar10',)
SEEDS = (0, 1, 2)
MINI_FFC = 60

rows = []
for method, dataset, proxy, seed in itertools.product(METHODS, DATASETS, PROXIES, SEEDS):
    ev, best = run_method(method, proxy=proxy, dataset=dataset,
                          ffc=MINI_FFC, seed=seed)
    rows.append({
        'method': method, 'dataset': dataset, 'proxy': proxy, 'seed': seed,
        'best_proxy': max(ev.best_score_per_ffc),
        'best_test_acc': ev.get_test_accuracy(best),
    })
df = pd.DataFrame(rows)
summary = df.groupby(['method', 'dataset', 'proxy']).agg(
    best_proxy_mean=('best_proxy', 'mean'),
    best_proxy_std=('best_proxy', 'std'),
    test_acc_mean=('best_test_acc', 'mean'),
).reset_index()
summary

---

**Scaling up to the paper's 84-cell main matrix**

From the project root, run

```bash
python -m scripts.run_main_matrix --workers 8 --seeds 100
```

Phase 1 above already downloaded everything this CLI needs. ImageNet-16-120 cells additionally require the NAS-Bench-201 ImageNet16 archive on disk; pass `--data_source imagenet16 --imagenet16_root <path>` for that dataset.
